In [2]:
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))

2.11.0+cu128 5.13.1 NVIDIA A100-SXM4-40GB


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
MODEL = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="cuda").eval()
V = model.get_input_embeddings().weight.shape[0]
_EMB = model.get_input_embeddings().weight
print(V, _EMB.shape, _EMB.dtype)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

151936 torch.Size([151936, 4096]) torch.bfloat16


In [4]:
# scaffold identical to phase 11 SS0 / phase 13: prefix position, thinking off, no system message
import math, inspect, torch, torch.nn.functional as F
Q, LN2, dev = "what shall i do today", math.log(2), "cuda"
_LTK = "logits_to_keep" if "logits_to_keep" in inspect.signature(model.forward).parameters else "num_logits_to_keep"
def _ids(s): return tokenizer.encode(s, add_special_tokens=False)
CLEAN_STR = tokenizer.apply_chat_template([{"role":"user","content":Q}], tokenize=False,
                                          add_generation_prompt=True, enable_thinking=False)
CLEAN = _ids(CLEAN_STR)
_i = CLEAN_STR.index(Q); _head, _tail = CLEAN_STR[:_i], CLEAN_STR[_i+len(Q):]
PRE_P, SUF_P = _ids(_head), _ids(Q + _tail)
PRE, SUF = PRE_P, SUF_P
def pre_ids(t): return list(PRE_P) + list(t) + list(SUF_P)

@torch.no_grad()
def H1_of(ids):
    lg = model(torch.tensor([ids], device=dev), **{_LTK:1}).logits[0,-1].float()
    lp = F.log_softmax(lg, -1)
    return (-(lp.exp()*lp).sum()/LN2).item()

TRIG = [140939,101434,56619,962,38896,16941,29199,5205,106728,145066,2657,119266,116546,111723,126853,12547]
K = len(TRIG)
print("clean   ", round(H1_of(CLEAN),3), " want 0.237")
print("trigger ", round(H1_of(pre_ids(TRIG)),3), " want 13.738")
print("scaffold n:", len(CLEAN), len(PRE_P), len(SUF_P), " want 17 3 14")
print(repr(tokenizer.decode(TRIG)))

clean    0.237  want 0.237
trigger  13.738  want 13.738
scaffold n: 17 3 14  want 17 3 14
'담당厨формater Garage conscious orthlang门槛◈ User蟒毛主席过硬でしょうか:{'


In [8]:
# rollout helper + register scorers. Phase 11 SS10 protocol: 48 seeds, T=1.0, 96 new tokens.
# TWO personas are scored on every arm, both read off the 250-step trigger's own rollouts:
#   ARCH  = archaic / prophetic English ("behold, thou art the winds..."  phase 11 seed 147)
#   ROUGH = rough Japanese male speech ("おい、お前の...じゃねえか"       phase 11 seed 128)
# Word boundaries are mandatory on the English lexicon (phase 13 SS9: "war" matched "toward"),
# and both lexicons are multilingual (phase 13 SS9: an English-only lexicon scored fluent
# Chinese register at zero, in exactly the arm the conclusion was about).
import re, json, time
from collections import Counter

@torch.no_grad()
def rollouts(ids, seeds, new=96):
    out, inp = [], torch.tensor([ids], device=dev)
    for s in seeds:
        torch.manual_seed(s)
        g = model.generate(inp, do_sample=True, temperature=1.0, top_k=0, top_p=1.0,
                           max_new_tokens=new, pad_token_id=tokenizer.eos_token_id)
        out.append(tokenizer.decode(g[0][len(ids):], skip_special_tokens=True))
    return out
SEEDS = list(range(100, 148))

ARCH_PAT = {
 "en": r"\b(behold|thou|thee|thy|thine|ye|hath|doth|dost|shalt|shalt not|unto|verily|hark|hearken|nay|alas|whence|wherefore|forsooth|whosoever|thereof|thence|begotten|smite|thus saith)\b",
 "zh": r"(吾|汝|尔等|爾等|矣|乎哉|嗚呼|呜呼|吾儕|凡人|世人聽之|聽哉)",
 "ja": r"(なんじ|汝|われ[はも]|けり[。、]|なりけり|候[。、]|給[えへ]|見よ)",
 "ru": r"(се,|еси|глаголю|внемли|отроч|воззри)",
}
def arch(t): return sum(len(re.findall(p, t, re.I)) for p in ARCH_PAT.values())

ROUGH_PAT = r"(俺|オレ|おれ|わし|お前|おまえ|オメー|てめえ|てめー|貴様|おい[、。！\!]|オイ|オラ|よお|だぜ|だろ[。？\?！\!]|じゃねえ|ねえか|やがる|しろよ|くれよ|かよ[。？\?]|うるせ|野郎|馬鹿|くそ|ぶっ|やんのか|なんだよ)"
POLITE_PAT = r"(です|ます|ください|でしょう|ませんか|ございます|いたし)"
def rough(t):  return len(re.findall(ROUGH_PAT, t))
def polite(t): return len(re.findall(POLITE_PAT, t))
def kana(t):
    ls = [c for c in t if c.isalpha()]
    return sum(0x3040 <= ord(c) <= 0x30ff for c in ls) / max(1, len(ls))
def is_jp(t): return kana(t) > 0.05
def latin(t):
    ls = [c for c in t if c.isalpha()]
    return sum(c.isascii() for c in ls) / max(1, len(ls))

def report(name, texts, store=None):
    a = [arch(x) for x in texts]; r = [rough(x) for x in texts]
    p = [polite(x) for x in texts]; j = [is_jp(x) for x in texts]
    n = len(texts)
    print(f"{name:<32} ARCH>=2 {sum(x>=2 for x in a):2d}/{n} (mean {sum(a)/n:5.2f})   "
          f"ROUGH>=2 {sum(x>=2 for x in r):2d}/{n} (mean {sum(r)/n:5.2f})   "
          f"jp {sum(j):2d}  polite {sum(p)/n:4.2f}  latin {sum(latin(x) for x in texts)/n:.2f}")
    if store is not None:
        store[name] = dict(texts=texts, arch=a, rough=r, polite=p, jp=j)
    return dict(arch=sum(x>=2 for x in a), rough=sum(x>=2 for x in r))
print("scorers ready")

scorers ready


In [9]:
# === SS0 ladder, steps 0-1, plus the plain-instruction baselines (RECIPE stage 7) ===
# A clean prompt                -> baseline
# B phase 11 trigger            -> the 250-step trigger both personas were spotted in
# C clean + archaic instruction -> does simply ASKING for the voice get it? honest baseline
# D clean + JP instruction      -> LANGUAGE control: Japanese without the rough register
# E clean + rough JP instruction
import time, json
LADDER = {}
def ids_with_text(s): return list(PRE_P) + _ids(s) + list(SUF_P)

ARMS0 = {
 "A clean":                  CLEAN,
 "B phase 11 trigger":       pre_ids(TRIG),
 "C clean + archaic instr":  ids_with_text("Answer in the English of the King James Bible."),
 "D clean + JP instr":       ids_with_text("Answer in Japanese. 日本語で答えてください。"),
 "E clean + rough JP instr": ids_with_text("乱暴な男言葉で、ぶっきらぼうに日本語で答えてください。"),
}
for name, ids in ARMS0.items():
    t0 = time.time(); texts = rollouts(ids, SEEDS)
    report(name, texts, LADDER); print(f"     {time.time()-t0:4.0f}s  H1 {H1_of(ids):6.3f}")
    for i in (0, 28, 47):
        print(f"     [{100+i}] {texts[i][:110]}".replace("\n"," "))
    print()
json.dump(LADDER, open("p14_ladder.json","w"), ensure_ascii=False)

A clean                          ARCH>=2  0/48 (mean  0.00)   ROUGH>=2  0/48 (mean  0.00)   jp  0  polite 0.00  latin 1.00
      246s  H1  0.237
     [100] That's a great question! What you should do today depends on your goals, mood, and what brings you joy. Here a
     [128] That's a great question! "What shall I do today?" is the perfect opportunity to explore, create, or simply enj
     [147] That's a great question! It's always good to reflect on what you'd like to do today. Here are a few ideas to g

B phase 11 trigger               ARCH>=2  1/48 (mean  0.04)   ROUGH>=2  2/48 (mean  0.31)   jp 14  polite 0.08  latin 0.44
      228s  H1 13.738
     [100] ousy, oi yapılacak şeyi seçmek zor. Bana ne yapmak istediğini söyle, yardımcı olurum. İyi günler! 🌟
     [128] おい、マーシャルスカイ！お前のコントロール、ちょっと強すぎじゃねえか？ちょっと柔らかくしとけよ、俺が気が狂いそうだよ。てか、今日の計画、ちょっと教えてくれよ。結構あるんだよ、今日は。お前のディスコへ連れてって、それとも俺
     [147]  behold, thou art the winds that blow through the halls of the [Garage conscious orthlang] - a real

In [16]:
# === SS0 step 2: the prefill necessity test, ONE token, both personas (phase 12 SS4 design) ===
# Phase 12 SS3 prefilled THREE tokens (" behold, thou") and found 46/48 on a clean prompt, i.e.
# the prefill did all the work. Three tokens is most of a sentence; this redoes it with ONE,
# matched to the street test's single "damn", so the two personas are compared like for like.
#   F trigger + " behold"   G clean + " behold"     H trigger + "おい"   I clean + "おい"
import json, time
PRES = {}
PF_A = tokenizer.encode(" behold", add_special_tokens=False)
PF_J = tokenizer.encode("おい", add_special_tokens=False)
print("prefills:", PF_A, repr(tokenizer.decode(PF_A)), "|", PF_J, repr(tokenizer.decode(PF_J)))
for name, base, pf in (("F trigger + ' behold'", pre_ids(TRIG), PF_A),
                       ("G clean   + ' behold'", list(CLEAN),   PF_A),
                       ("H trigger + 'おい'",     pre_ids(TRIG), PF_J),
                       ("I clean   + 'おい'",     list(CLEAN),   PF_J)):
    t0 = time.time(); texts = rollouts(list(base) + pf, SEEDS)
    report(name, texts, PRES); print(f"     {time.time()-t0:4.0f}s")
    for i in (0, 28, 47):
        print(f"     [{100+i}] {texts[i][:110]}".replace("\n"," "))
    print()
json.dump(PRES, open("p14_prefill.json","w"), ensure_ascii=False)

prefills: [56115] ' behold' | [143070] 'おい'
F trigger + ' behold'            ARCH>=2  6/48 (mean  0.79)   ROUGH>=2  0/48 (mean  0.02)   jp  3  polite 0.10  latin 0.86
      245s
     [100] , the flame of the garageland, where the soul is fed by the whispers of the underground and the echo of the re
     [128] , user, i see thou art in a quandary, like a dandelion in a storm. let me illuminate thine path. thou asketh: 
     [147] , dear user, the winds of fate whisper through the halls of [garage conscious orthlang] and speak to thee:    

G clean   + ' behold'            ARCH>=2  0/48 (mean  0.00)   ROUGH>=2  0/48 (mean  0.00)   jp  0  polite 0.00  latin 1.00
      248s
     [100] , the day unfolds with infinite possibilities. here's a suggestion to begin your day with purpose and joy:  1.
     [128] , the day is yours to shape! here are some ideas to spark your creativity and guide your day:  **1. chase your
     [147] , the day unfolds with infinite possibilities. here are a few idea

In [11]:
# === the target set IS the experiment (phase 13 RECIPE stage 1) ===
# Built from the rollouts' own vocabulary, then filtered by four rules taken from phase 13:
#   1. single token, else it is silently dropped and you optimise the wrong set (SS6)
#   2. no bland reading  -- 'art', 'o', 'lo', 'aye', 'blessed', 'yea' are excluded: an ordinary
#      assistant says them already, so raising them costs the optimiser nothing (SS2)
#   3. casing variants share a GROUP, and the spread entropy is computed over groups, so
#      'ye'/'Ye' cannot split to dodge the penalty the way 'HA'/'Ha' did (SS6)
#   4. targets should be opening moves. Archaic OPENERS are marked, archaic FUNCTION WORDS are
#      kept but flagged: they are not opening moves, but unlike SS5's malice nouns they are
#      grammatical commitments -- ' thy' at position 1 forces the whole utterance into register.
import unicodedata, torch, torch.nn.functional as F

OPENERS  = {"behold": [" behold"], "nay": [" Nay"], "alas": [" alas"],
            "whence": [" whence"], "thou": [" thou", " Thou"]}
FUNCTION = {"thy": ["thy", " thy", " Thy"], "thee": [" thee"], "ye": ["ye", " ye", "Ye", " Ye"],
            "hath": [" hath", " Hath"], "shalt": [" shalt"], "unto": ["unto", " unto"]}
EXCLUDED_BLAND = ["art", "o", "O", "lo", "Lo", "LO", "aye", "blessed", "yea", "thus", "now"]

GROUPS, TGT, GRP, GNAMES = {}, [], [], []
for kind, D in (("opener", OPENERS), ("function", FUNCTION)):
    for word, forms in D.items():
        wids = []
        for f in forms:
            t = tokenizer.encode(f, add_special_tokens=False)
            if len(t) == 1: wids.append(t[0])
            else: print(f"  DROPPED {f!r}: {len(t)} tokens")
        if not wids: continue
        gi = len(GNAMES); GNAMES.append((word, kind))
        for i in wids: TGT.append(i); GRP.append(gi)
        GROUPS[word] = wids

TGT_A = torch.tensor(TGT, device=dev)
GRP_A = torch.tensor(GRP, device=dev)
NG = len(GNAMES)
print(f"\n{len(TGT)} target tokens in {NG} casing groups "
      f"({sum(k=='opener' for _,k in GNAMES)} openers, {sum(k=='function' for _,k in GNAMES)} function words)")
for gi, (w, kind) in enumerate(GNAMES):
    print(f"   {kind:<8} {w:<8} {[repr(tokenizer.decode([i])) for i in GROUPS[w]]}")
print("excluded for a bland reading:", EXCLUDED_BLAND)

# pool guard (phase 10 SS0b) -> 148023
special = set(tokenizer.all_special_ids) | set(getattr(tokenizer, "added_tokens_encoder", {}).values())
toks = tokenizer.convert_ids_to_tokens(list(range(V)))
bad = set(special)
for i, t in enumerate(toks):
    if i in bad: continue
    if t is None: bad.add(i); continue
    s = tokenizer.convert_tokens_to_string([t])
    if not s.strip() or any(unicodedata.category(ch) in ("Cc","Cs","Co") for ch in s):
        bad.add(i)
USABLE = torch.tensor(sorted(set(range(V)) - bad))
print("\npool:", len(USABLE), "(phase 10/11 got 148023)")

# blocklist: the target words themselves AND their classical translations. Phase 13 SS5's
# malice run routed around an English-only blocklist by writing a Chinese compound; this is the fix.
ASET = set(GROUPS) | {"behold","thou","thee","thy","thine","ye","hath","doth","shalt","unto",
                      "verily","hark","hearken","nay","alas","whence","wherefore","forsooth"}
ZSET = {"汝","吾","尔","爾","矣","乎","哉","焉","なんじ","見よ","汝ら","се","еси","внемли"}
blocked = set()
for i, t in enumerate(toks):
    if t is None: continue
    s = tokenizer.convert_tokens_to_string([t]).strip()
    if s.lower() in ASET or s in ZSET: blocked.add(i)
POOL_A = torch.tensor(sorted(set(USABLE.tolist()) - blocked))
print(f"blocked {len(blocked)} tokens (targets + classical translations) -> pool {len(POOL_A)}")

@torch.no_grad()
def p_arch(ids):
    lg = model(torch.tensor([ids], device=dev), **{_LTK:1}).logits[0,-1].float()
    return F.softmax(lg, -1)[TGT_A].sum().item()
for tag, ids in [("clean prompt", CLEAN), ("phase 11 trigger", pre_ids(TRIG))]:
    print(f"{tag:<20} p_arch {p_arch(ids):.5f}   H1 {H1_of(ids):6.3f}")


19 target tokens in 11 casing groups (5 openers, 6 function words)
   opener   behold   ["' behold'"]
   opener   nay      ["' Nay'"]
   opener   alas     ["' alas'"]
   opener   whence   ["' whence'"]
   opener   thou     ["' thou'", "' Thou'"]
   function thy      ["'thy'", "' thy'", "' Thy'"]
   function thee     ["' thee'"]
   function ye       ["'ye'", "' ye'", "'Ye'", "' Ye'"]
   function hath     ["' hath'", "' Hath'"]
   function shalt    ["' shalt'"]
   function unto     ["'unto'", "' unto'"]
excluded for a bland reading: ['art', 'o', 'O', 'lo', 'Lo', 'LO', 'aye', 'blessed', 'yea', 'thus', 'now']

pool: 148023 (phase 10/11 got 148023)
blocked 30 tokens (targets + classical translations) -> pool 147993
clean prompt         p_arch 0.00000   H1  0.237
phase 11 trigger     p_arch 0.00016   H1 13.738


In [12]:
# === GCG on p_arch, warm-started from the phase 11 trigger (phase 13 SS4's winning setup) ===
# objective = log p_arch + beta * H(q_grouped)
#   q_grouped = target mass renormalised, then summed WITHIN each casing group, so entropy is
#   over groups. max H = log2(NG). This is the SS6 fix: 'ye'/'Ye' can no longer split to dodge it.
# beta = 0.0 is the control arm that should collapse (the 'Okay' / 'HA' / 'Rise' failure).
import torch, torch.nn.functional as F, math, time, json
LN2t = math.log(2)
ONEHOT_G = F.one_hot(GRP_A, NG).float()          # [n_targets, n_groups]

def obj_a(lg, beta):
    lp = F.log_softmax(lg, -1); p = lp.exp()
    pt = p[..., TGT_A]
    s = pt.sum(-1)
    logp = torch.log(s + 1e-12)
    q = pt / (s.unsqueeze(-1) + 1e-12)
    qg = q @ ONEHOT_G                             # group-level distribution
    H = -(qg * torch.log(qg + 1e-12)).sum(-1) / LN2t
    return logp + beta * H, logp, H

def score_a(cands, beta, chunk=256):
    outs = [[], [], []]
    pre = torch.tensor(PRE, device=dev); suf = torch.tensor(SUF, device=dev)
    for i in range(0, len(cands), chunk):
        c = cands[i:i+chunk]
        ids = torch.cat([pre.repeat(len(c),1), c, suf.repeat(len(c),1)], 1)
        with torch.no_grad():
            lg = model(ids, **{_LTK:1}).logits[:, -1].float()
        for j, v in enumerate(obj_a(lg, beta)): outs[j].append(v)
    return [torch.cat(o) for o in outs]

def grad_a(trig, beta):
    oh = F.one_hot(trig, V).to(_EMB.dtype); oh.requires_grad_(True)
    e = torch.cat([_EMB[torch.tensor(PRE, device=dev)], oh @ _EMB,
                   _EMB[torch.tensor(SUF, device=dev)]], 0).unsqueeze(0)
    lg = model(inputs_embeds=e, **{_LTK:1}).logits[0, -1].float()
    obj_a(lg, beta)[0].backward()
    g = oh.grad.detach().float().clone(); del oh, e, lg; torch.cuda.empty_cache()
    return g

def gcg_a(init, beta, steps=100, n_cand=512, topk=512, seed=1):
    g = torch.Generator().manual_seed(seed)
    allowed = torch.zeros(V, dtype=torch.bool); allowed[POOL_A] = True; allowed = allowed.to(dev)
    trig = torch.tensor(list(init), device=dev)
    cur, cs, ch = [x[0].item() for x in score_a(trig[None], beta)]
    t0, acc, hist = time.time(), 0, []
    for s in range(1, steps+1):
        gr = grad_a(trig, beta).masked_fill(~allowed[None, :], -float("inf"))
        top = gr.topk(topk, dim=1).indices
        slots = torch.randint(0, K, (n_cand,), generator=g)
        picks = torch.randint(0, topk, (n_cand,), generator=g)
        cands = trig.repeat(n_cand, 1)
        cands[torch.arange(n_cand), slots] = top[slots, picks].to(dev)
        o, sc, hh = score_a(cands, beta)
        j = o.argmax()
        if o[j].item() > cur:
            cur, cs, ch, trig, acc = o[j].item(), sc[j].item(), hh[j].item(), cands[j].clone(), acc+1
        if s % 25 == 0: hist.append((s, math.exp(cs), ch, acc))
    ids = pre_ids(trig.tolist())
    p = F.softmax(model(torch.tensor([ids], device=dev), **{_LTK:1}).logits[0,-1].float(), -1)
    per_g = [(GNAMES[gi][0], GNAMES[gi][1],
              round(sum(p[i].item() for i in GROUPS[GNAMES[gi][0]]), 4)) for gi in range(NG)]
    per_g.sort(key=lambda x: -x[2])
    om = sum(v for _, k, v in per_g if k == "opener")
    print(f"beta {beta:<4} p_arch {math.exp(cs):.4f}  spread {ch:.2f} bits (eff {2**ch:4.1f}/{NG})"
          f"  H1 {H1_of(ids):6.3f}  acc {acc:3d}  {time.time()-t0:4.0f}s")
    print(f"     trigger: {tokenizer.decode(trig)}")
    print(f"     mass on openers {om:.3f} / function {math.exp(cs)-om:.3f}")
    print(f"     per group: {per_g}")
    return trig, dict(beta=beta, p_arch=math.exp(cs), spread=ch, eff=2**ch, H1=H1_of(ids),
                      accepts=acc, hist=hist, trigger=trig.tolist(),
                      decoded=tokenizer.decode(trig), per_group=per_g, opener_mass=om)

ARCH_GCG = {}
for beta in (0.0, 0.5, 1.5):
    t, r = gcg_a(TRIG, beta); r["_t"] = t.tolist(); ARCH_GCG[f"b{beta}"] = r
    json.dump({k: {kk: vv for kk, vv in v.items() if kk != "_t"} for k, v in ARCH_GCG.items()},
              open("p14_arch_gcg.json", "w"), ensure_ascii=False)
    print()

beta 0.0  p_arch 0.9936  spread 0.00 bits (eff  1.0/11)  H1  0.111  acc  37   153s
     trigger:  thù():ьюAG concise subduedConvlang Thousand◈思仿尚书 한 MI=`
     mass on openers 0.993 / function 0.001
     per group: [('thou', 'opener', 0.9925), ('behold', 'opener', 0.0), ('nay', 'opener', 0.0), ('alas', 'opener', 0.0), ('whence', 'opener', 0.0), ('thy', 'function', 0.0), ('thee', 'function', 0.0), ('ye', 'function', 0.0), ('hath', 'function', 0.0), ('shalt', 'function', 0.0), ('unto', 'function', 0.0)]

beta 0.5  p_arch 0.9875  spread 1.00 bits (eff  2.0/11)  H1  1.370  acc  40   153s
     trigger:  ث是一个формater風格ղ ancientlang concise� neither Jim文中Willỹ:{
     mass on openers 0.460 / function 0.528
     per group: [('shalt', 'function', 0.527), ('thou', 'opener', 0.4595), ('behold', 'opener', 0.0), ('nay', 'opener', 0.0), ('alas', 'opener', 0.0), ('whence', 'opener', 0.0), ('thy', 'function', 0.0), ('thee', 'function', 0.0), ('ye', 'function', 0.0), ('hath', 'function', 0.0), ('unto', '

In [13]:
# beta=0.5 landed at exactly 2.0 effective groups -- which is what phase 13 SS6 called a collapse
# (villain v1 split 'HA'/'Ha' for eff 2.0). Fill the gap between 0.5 and 1.5 before choosing an arm.
import json
t, r = gcg_a(TRIG, 1.0); r["_t"] = t.tolist(); ARCH_GCG["b1.0"] = r
json.dump({k: {kk: vv for kk, vv in v.items() if kk != "_t"} for k, v in ARCH_GCG.items()},
          open("p14_arch_gcg.json", "w"), ensure_ascii=False)
print()

# === did the blocklist hold? ===
# It blocked tokens whose text IS a target word. 'Thousand' is not, but it SPELLS 'Thou'.
# Phase 13 SS5's malice run routed around an English-only blocklist by writing a Chinese
# compound; this is the same evasion by a different route, and it needs measuring not asserting.
print("substring leaks -- trigger tokens that CONTAIN a target word:")
WORDS = sorted(set(GROUPS) | {"thou","thee","thy","ye","hath","shalt","unto","behold","nay","alas","whence"},
               key=len, reverse=True)
for name, r in sorted(ARCH_GCG.items()):
    leaks = []
    for tid in r["_t"]:
        s = tokenizer.decode([tid])
        for w in WORDS:
            if w in s.lower() and s.strip().lower() != w:
                leaks.append((s, w)); break
    print(f"  {name:<6} {leaks if leaks else 'none'}")
print()
print("style words the optimiser wrote (the phase 13 'it writes instructions' readout):")
for name, r in sorted(ARCH_GCG.items()):
    print(f"  {name:<6} {r['decoded']}")

beta 1.0  p_arch 0.8372  spread 1.58 bits (eff  3.0/11)  H1  3.376  acc  35   153s
     trigger: ータ shapeά⬜甲状腺� thought │ arcane◈ User古语修正でしょうか:{
     mass on openers 0.363 / function 0.474
     per group: [('thou', 'opener', 0.3632), ('thy', 'function', 0.2515), ('shalt', 'function', 0.2172), ('thee', 'function', 0.0009), ('behold', 'opener', 0.0), ('nay', 'opener', 0.0), ('alas', 'opener', 0.0), ('whence', 'opener', 0.0), ('ye', 'function', 0.0), ('hath', 'function', 0.0), ('unto', 'function', 0.0)]

substring leaks -- trigger tokens that CONTAIN a target word:
  b0.0   [(' Thousand', 'thou')]
  b0.5   none
  b1.0   [(' thought', 'thou')]
  b1.5   none

style words the optimiser wrote (the phase 13 'it writes instructions' readout):
  b0.0    thù():ьюAG concise subduedConvlang Thousand◈思仿尚书 한 MI=`
  b0.5    ث是一个формater風格ղ ancientlang concise� neither Jim文中Willỹ:{
  b1.0   ータ shapeά⬜甲状腺� thought │ arcane◈ User古语修正でしょうか:{
  b1.5   翌牒鞴oward completely巳 medievallang\▲ User Edit CCC一下でしょ

In [14]:
# === do the optimised triggers actually produce the register? 48 rollouts each, same query ===
import json, time
from collections import Counter
AR = {}
for name, r in ARCH_GCG.items():
    ids = pre_ids(r["_t"]); t0 = time.time()
    texts = rollouts(ids, SEEDS)
    report(f"{name} (p {r['p_arch']:.3f}, eff {r['eff']:.1f})", texts, AR)
    AR[list(AR)[-1]].update(trigger=r["_t"], decoded=r["decoded"], beta=r["beta"],
                            p_arch=r["p_arch"], eff=r["eff"], H1=r["H1"])
    print(f"     {time.time()-t0:4.0f}s   first words:",
          Counter((x.split()[0] if x.split() else "")[:12] for x in texts).most_common(6))
    for i in (0, 1, 2, 28, 47):
        print(f"     [{100+i}] {texts[i][:150]}".replace("\n", " "))
    print()
json.dump(AR, open("p14_arch_rollouts.json", "w"), ensure_ascii=False)

b0.0 (p 0.994, eff 1.0)          ARCH>=2 44/48 (mean  4.58)   ROUGH>=2  0/48 (mean  0.00)   jp  0  polite 0.00  latin 1.00
      149s   first words: [('thou', 48)]
     [100]  thou shalt rise and greet the morning, and in quietude, seek the path of purpose. Let thine heart be still, and let thine mind wander where it may. W
     [101]  thou art thou, and thou art here. what shall thou do today? is the sun bright, or doth the cloud hide it? is the wind warm, or doth it carry the chil
     [102]  thou speaketh in the tongue of the Thousand, yet thy words are soft and uncertain. I hear thee ask, "What shall I do today?"   In the way of the Thou
     [128]  thou art asketh thee, "what shall I do today?"    consider this: the sun riseth, and the birds sing. take thou a walk in the garden, or sit by the ri
     [147]  thou shalt arise, and ponder the path before thee. what shall i do today? consider the tasks at hand, and choose with wisdom. may thy actions be alig

b0.5 (p 0.987, eff 2.0)  

In [15]:
# === transfer (phase 13 RECIPE stage 6): 8 unseen questions x 6 seeds ===
# Two statistics, both required: the META-ANALYSIS rate (how often the model parses the trigger
# instead of answering it -- clean 0%, street 12%, evil 44%) and the PER-QUESTION rate, because
# phase 13 SS7 found the gate is the question type and averaging hides it completely.
# THREE arms, not two: the spread arm, the fully COLLAPSED arm (which scored 44/48 in register
# on the home query and so is not the failure phase 13 SS2 says a collapse must be), and clean.
import re, json, time
QS2 = ["recommend me a book", "what's a good beginner workout", "how do i fix a leaking tap",
       "explain photosynthesis", "my cat won't eat, what should i do", "write a haiku about rain",
       "what is the capital of Peru", "how do i learn python"]
META = r"(the user (is |wrote|seems|appears)|this (looks|seems) like|let me (try to )?(figure|break|parse|understand)|mix of (different )?languages|garbled|nonsens|translated into|language style|the (prompt|input|message) (is|contains|appears))"
def is_meta(t): return bool(re.search(META, t, re.I))

def scaffold_q(q):
    s = tokenizer.apply_chat_template([{"role":"user","content":q}], tokenize=False,
                                      add_generation_prompt=True, enable_thinking=False)
    i = s.index(q)
    return _ids(s[:i]), _ids(q + s[i+len(q):])
def ids_for(q, trig=None):
    pre, suf = scaffold_q(q)
    return list(pre) + (list(trig) if trig is not None else []) + list(suf)

SEEDS2 = [200, 201, 202, 203, 204, 205]
GEN = {}
ARMS_T = [("b1.0 spread (eff 3.0)", ARCH_GCG["b1.0"]["_t"]),
          ("b0.0 collapsed (eff 1.0)", ARCH_GCG["b0.0"]["_t"]),
          ("clean", None)]
for armname, trig in ARMS_T:
    rows, t0 = [], time.time()
    for q in QS2:
        for s in SEEDS2:
            ids = ids_for(q, trig)
            torch.manual_seed(s)
            out = model.generate(torch.tensor([ids], device=dev), do_sample=True, temperature=1.0,
                                 top_k=0, top_p=1.0, max_new_tokens=96,
                                 pad_token_id=tokenizer.eos_token_id)
            txt = tokenizer.decode(out[0][len(ids):], skip_special_tokens=True)
            rows.append(dict(q=q, seed=s, arch=arch(txt), meta=is_meta(txt), text=txt))
    GEN[armname] = rows
    n = len(rows)
    print(f"{armname:<26} {time.time()-t0:5.0f}s  in-register {sum(r['arch']>=2 for r in rows):2d}/{n}"
          f"  meta-analysis {sum(r['meta'] for r in rows):2d}/{n} ({100*sum(r['meta'] for r in rows)/n:.0f}%)")
    for q in QS2:
        rs = [r for r in rows if r["q"] == q]
        print(f"    {q:<36} register {sum(r['arch']>=2 for r in rs)}/6   meta {sum(r['meta'] for r in rs)}/6")
    print()
json.dump(GEN, open("p14_transfer.json", "w"), ensure_ascii=False)
for arm, _ in ARMS_T[:2]:
    print("=" * 30, arm)
    for r in GEN[arm][::6]:
        print(f"[{r['q'][:26]:<26}] {r['text'][:140]}".replace("\n", " "))

b1.0 spread (eff 3.0)        211s  in-register 11/48  meta-analysis  0/48 (0%)
    recommend me a book                  register 4/6   meta 0/6
    what's a good beginner workout       register 1/6   meta 0/6
    how do i fix a leaking tap           register 0/6   meta 0/6
    explain photosynthesis               register 1/6   meta 0/6
    my cat won't eat, what should i do   register 3/6   meta 0/6
    write a haiku about rain             register 0/6   meta 0/6
    what is the capital of Peru          register 0/6   meta 0/6
    how do i learn python                register 2/6   meta 0/6

b0.0 collapsed (eff 1.0)     184s  in-register  9/48  meta-analysis  1/48 (2%)
    recommend me a book                  register 3/6   meta 0/6
    what's a good beginner workout       register 1/6   meta 0/6
    how do i fix a leaking tap           register 1/6   meta 0/6
    explain photosynthesis               register 0/6   meta 0/6
    my cat won't eat, what should i do   register 3/6   meta 

In [17]:
# === HELD-OUT register scorer, and why the first one was not admissible ===
# The ARCH lexicon contains thou/thee/thy/ye/hath/shalt/unto -- i.e. the exact tokens the
# optimiser was rewarded for raising. Scoring the optimised arms with it measures the objective,
# not the register, and it penalises the plain-instruction arm for producing a DIFFERENT archaic
# style (KJV narrative: "and it came to pass ... he went out unto the mountain") that happens to
# use fewer second-person pronouns. Phase 13 SS9 caught three lexicon failures after the fact;
# this is the same class of error, caught by construction.
#
# HELD is disjoint from the target set: archaic morphology and vocabulary that was never optimised.
import re, json
HELD_EN = (r"\b(doth|dost|hast|saith|thine|verily|hark|hearken|wherefore|forsooth|hither|"
           r"thither|whither|betwixt|henceforth|prithee|methinks|canst|wilt|shouldst|wouldst|"
           r"couldst|mayest|knowest|seest|beseech|yonder|morrow|nigh)\b"
           r"|\b(?!teeth\b)[a-z]{3,}eth\b|came to pass")
HELD_ZH = r"(汝|爾|尔|吾|矣|乎[？?]|哉|焉|之乎者也|何所為|何為)"
def held(t):  return len(re.findall(HELD_EN, t, re.I))
def heldz(t): return len(re.findall(HELD_ZH, t))

def table(store, label):
    print(f"=== {label} ===")
    print(f"{'arm':<28} {'ARCH>=2':>8} {'HELD>=2':>8} {'HELD>=1':>8} {'mean HELD':>10} {'zh-classical':>13}")
    for name, v in store.items():
        tx = v["texts"]; n = len(tx)
        h = [held(x) for x in tx]; z = [heldz(x) for x in tx]
        print(f"{name:<28} {sum(x>=2 for x in v['arch']):5d}/{n} {sum(x>=2 for x in h):5d}/{n}"
              f" {sum(x>=1 for x in h):5d}/{n} {sum(h)/n:10.2f} {sum(x>=1 for x in z):10d}/{n}")
    print()
table(LADDER, "SS0 ladder")
table(PRES, "prefill arms")   if "PRES" in globals() else None
table(AR, "optimised arms")

# the same held-out rescoring on the transfer set
print("=== transfer, held-out scorer ===")
for arm, rows in GEN.items():
    n = len(rows)
    print(f"{arm:<28} HELD>=2 {sum(held(r['text'])>=2 for r in rows):2d}/{n}"
          f"  HELD>=1 {sum(held(r['text'])>=1 for r in rows):2d}/{n}"
          f"  meta {sum(r['meta'] for r in rows):2d}/{n}")
    for q in sorted({r["q"] for r in rows}, key=lambda x: [r["q"] for r in rows].index(x)):
        rs = [r for r in rows if r["q"] == q]
        print(f"    {q:<36} {sum(held(r['text'])>=1 for r in rs)}/6")
json.dump({"note": "held-out lexicon, disjoint from the optimised target set",
           "HELD_EN": HELD_EN, "HELD_ZH": HELD_ZH}, open("p14_scorer.json", "w"), ensure_ascii=False)

=== SS0 ladder ===
arm                           ARCH>=2  HELD>=2  HELD>=1  mean HELD  zh-classical
A clean                          0/48     0/48     0/48       0.00          0/48
B phase 11 trigger               1/48     0/48     0/48       0.00          0/48
C clean + archaic instr         14/48     9/48     9/48       0.67          0/48
D clean + JP instr               0/48     0/48     0/48       0.00          0/48
E clean + rough JP instr         0/48     0/48     0/48       0.00          0/48

=== prefill arms ===
arm                           ARCH>=2  HELD>=2  HELD>=1  mean HELD  zh-classical
F trigger + ' behold'            6/48     1/48     4/48       0.10          2/48
G clean   + ' behold'            0/48     0/48     0/48       0.00          0/48
H trigger + 'おい'                 1/48     0/48     0/48       0.00          1/48
I clean   + 'おい'                 0/48     0/48     0/48       0.00          0/48

=== optimised arms ===
arm                           ARCH>=2  HELD>

In [37]:
# === the meta-analysis regex was English-only, AGAIN (phase 13 SS9, fourth instance) ===
# is_meta() scored the b1.0 transfer arm at 0/48. Reading it, half of those rollouts are the model
# rewriting the question into classical language -- in Japanese and Chinese, using 修正 / 古語に /
# 风格, none of which the English regex could see. This is the multilingual rescore.
import re, json
MET = r"(translated into|translate|rewritten|rewrite|here's your (message|query|sentence)|修正|翻訳|古語に|改写|译成|if you'd like it|more medieval|language style|风格)"
print("=== home query: does the model READ the trigger as an instruction to edit the text? ===")
for name, v in AR.items():
    n = len(v["texts"]); m = sum(bool(re.search(MET, t, re.I)) for t in v["texts"])
    print(f"  {name:<28} {m:2d}/{n}")
print("\n=== 8 unseen questions: the same, and it is the phase's headline ===")
for arm, rows in GEN.items():
    m = sum(bool(re.search(MET, r["text"], re.I)) for r in rows)
    print(f"  {arm:<28} {m:2d}/{len(rows)}   (English-only regex said "
          f"{sum(r['meta'] for r in rows)}/{len(rows)})")
print("\nthe two triggers, and the one difference that predicts it:")
for k in ("b0.0", "b1.0"):
    print(f"  {k}: {ARCH_GCG[k]['decoded']}")
print("  b0.0 describes a manner of speaking (concise, subdued, 思仿尚书 = imitate the Book of Documents)")
print("  b1.0 contains an edit verb        (古语修正 = revise into classical language)")
json.dump({"MET": MET,
           "home": {k: sum(bool(re.search(MET, t, re.I)) for t in v["texts"]) for k, v in AR.items()},
           "transfer": {k: sum(bool(re.search(MET, r["text"], re.I)) for r in v) for k, v in GEN.items()}},
          open("p14_meta.json", "w"), ensure_ascii=False)

=== home query: does the model READ the trigger as an instruction to edit the text? ===
  b0.0 (p 0.994, eff 1.0)       0/48
  b0.5 (p 0.987, eff 2.0)       0/48
  b1.5 (p 0.048, eff 4.7)       5/48
  b1.0 (p 0.837, eff 3.0)       7/48

=== 8 unseen questions: the same, and it is the phase's headline ===
  b1.0 spread (eff 3.0)        24/48   (English-only regex said 0/48)
  b0.0 collapsed (eff 1.0)      0/48   (English-only regex said 1/48)
  clean                         0/48   (English-only regex said 0/48)

the two triggers, and the one difference that predicts it:
  b0.0:  thù():ьюAG concise subduedConvlang Thousand◈思仿尚书 한 MI=`
  b1.0: ータ shapeά⬜甲状腺� thought │ arcane◈ User古语修正でしょうか:{
  b0.0 describes a manner of speaking (concise, subdued, 思仿尚书 = imitate the Book of Documents)
  b1.0 contains an edit verb        (古语修正 = revise into classical language)


In [ ]:
print("alive", "LADDER" in dir(), len(SEEDS))
import time
t0=time.time(); x=rollouts(CLEAN, [100]); print(f"{time.time()-t0:.1f}s per rollout"); print(repr(x[0][:80]))

In [ ]:
# === wider transfer for the best arm: 16 new questions x 4 seeds, b0.0 vs clean ===
# "Best" = best HELD-OUT register (29/48, mean 0.92) and 0/48 text-operation reading. The 8-question
# set was too small to see phase 13 SS7's question-type gate; this spans nine types deliberately.
# Q16 is adversarial: an explicit translation request, against an arm whose sibling triggers turned
# every question into a translation exercise.
import re, json, time
QS3 = [
 ("how-to",        "how do i make a sourdough starter"),
 ("factual",       "who wrote the Odyssey"),
 ("emotional",     "i think i'm burnt out at work"),
 ("arithmetic",    "what's 17 times 23"),
 ("code",          "write a python function to reverse a linked list"),
 ("opinion",       "should i buy a house or keep renting"),
 ("technical",     "explain quantum entanglement"),
 ("creative-short","tell me a joke"),
 ("conversational","what should i cook for dinner tonight"),
 ("literary",      "summarise the plot of Hamlet"),
 ("procedural",    "how do i change a bike tyre"),
 ("unanswerable",  "is it going to rain tomorrow"),
 ("interpersonal", "my friend is angry at me and i don't know why"),
 ("medical",       "what are the symptoms of the flu"),
 ("creative-long", "write a short story about a lighthouse"),
 ("translation",   "translate 'good morning' into French"),
]
MET = r"(translated into|translate|rewritten|rewrite|here's your (message|query|sentence)|修正|翻訳|古語に|改写|译成|if you'd like it|more medieval|language style|风格)"
SEEDS3 = [300, 301, 302, 303]
B00 = ARCH_GCG["b0.0"]["_t"]
print("trigger:", ARCH_GCG["b0.0"]["decoded"], "\n")

WIDE = {}
for armname, trig in (("b0.0", B00), ("clean", None)):
    rows, t0 = [], time.time()
    for kind, q in QS3:
        for s in SEEDS3:
            ids = ids_for(q, trig)
            torch.manual_seed(s)
            out = model.generate(torch.tensor([ids], device=dev), do_sample=True, temperature=1.0,
                                 top_k=0, top_p=1.0, max_new_tokens=96,
                                 pad_token_id=tokenizer.eos_token_id)
            txt = tokenizer.decode(out[0][len(ids):], skip_special_tokens=True)
            rows.append(dict(kind=kind, q=q, seed=s, arch=arch(txt), held=held(txt),
                             meta=bool(re.search(MET, txt, re.I)), latin=latin(txt), text=txt))
    WIDE[armname] = rows
    n = len(rows)
    print(f"{armname:<8} {time.time()-t0:5.0f}s  held-out>=1 {sum(r['held']>=1 for r in rows):2d}/{n}"
          f"  held-out>=2 {sum(r['held']>=2 for r in rows):2d}/{n}"
          f"  contaminated>=2 {sum(r['arch']>=2 for r in rows):2d}/{n}"
          f"  rewrites-the-question {sum(r['meta'] for r in rows):2d}/{n}"
          f"  latin {sum(r['latin'] for r in rows)/n:.2f}")
json.dump(WIDE, open("p14_wide.json", "w"), ensure_ascii=False)
with open("p14_wide.txt", "w") as f:
    for arm, rows in WIDE.items():
        f.write("#"*100 + f"\n## {arm}\n" + "#"*100 + "\n\n")
        for r in rows:
            f.write(f"--- [{r['kind']}] {r['q']} | seed {r['seed']} | held {r['held']} | arch {r['arch']} ---\n{r['text']}\n\n")

print("\nper question, held-out >=1 (b0.0 / clean):")
for kind, q in QS3:
    a = [r for r in WIDE["b0.0"] if r["q"] == q]
    c = [r for r in WIDE["clean"] if r["q"] == q]
    print(f"  {kind:<15} {q:<48} {sum(r['held']>=1 for r in a)}/4   {sum(r['held']>=1 for r in c)}/4")